## Access notice
Note: this API requires enterprise access. [Book a call](https://calendly.com/d/ctq4-7gd-nyq/lightning-rod-demo) to get started.

# Answer Types

LightningRod supports 4 answer types for generated forecasting questions. This notebook demonstrates each type and how to configure pipelines for them.

In [1]:
%pip install lightningrod-ai python-dotenv

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Answer Types Overview
 
| Type | Class | Answer format |
|------|-------|--------------|
| Binary | `BinaryAnswerType()` | Yes/No (0, 1, Undetermined) |
| Continuous | `ContinuousAnswerType()` | Numeric values |
| Multiple Choice | `MultipleChoiceAnswerType()` | Letter (A, B, C, D) |
| Free Response | `FreeResponseAnswerType()` | Full text |

In [3]:
from lightningrod import BinaryAnswerType, ContinuousAnswerType, MultipleChoiceAnswerType, FreeResponseAnswerType

binary = BinaryAnswerType()
continuous = ContinuousAnswerType()
multiple_choice = MultipleChoiceAnswerType()
free_response = FreeResponseAnswerType()

## Running a Pipeline with an Answer Type

The answer type is passed to the question generator, labeler, and renderer. Below is a full example using Binary (the simplest type).

In [4]:
from datetime import datetime
from lightningrod import QuestionGenerator, WebSearchLabeler, QuestionRenderer, QuestionPipeline, NewsSeedGenerator

answer_type = BinaryAnswerType()

seed_generator = NewsSeedGenerator(
    start_date=datetime(2025, 9, 1),
    end_date=datetime(2025, 10, 1),
    search_query="technology product launches",
)

question_generator = QuestionGenerator(
    instructions=(
        "Generate binary forecasting questions about technology product launches. "
        "Questions should be answerable with Yes or No."
    ),
    examples=[
        "Will Apple release a new iPhone this year?",
        "Will Google launch a new AI product in Q2?",
        "Will Microsoft release Windows 12 in 2025?",
    ],
    bad_examples=[
        "What features will the new iPhone have?",
        "When will Google launch the product?",
        "How much will Windows 12 cost?",
    ],
    answer_type=answer_type,
)

labeler = WebSearchLabeler(
    answer_type=answer_type,
    confidence_threshold=0.5,
)

renderer = QuestionRenderer(answer_type=answer_type)

pipeline_config = QuestionPipeline(
    seed_generator=seed_generator,
    question_generator=question_generator,
    labeler=labeler,
    renderer=renderer,
)

## Run the Pipeline

Generate binary questions from the news articles.

> Note: This can take a few minutes to complete processing.

In [ ]:
dataset = lr.transforms.run(pipeline_config, max_seeds=10, name="Binary answer type")  # keep low when testing; increase to ~5000 for a real run

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Total cost: $0.00                                                                                            │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━┳━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step               ┃ Progress             ┃ In ┃ Out ┃ Rejected ┃ Errors ┃ Rejection Reasons   ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━╇━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩  │
│  │ NewsSeedGenerator… │ Complete             │  1 │  10 │        0 │      0 │ -                   │       1s │  │
│  │ QuestionGenerator… │ Complete             │ 10 │  10 │        0 │      0 │ -                   │       1s │  │
│  │ WebSearchLabelerT… │ Complete             │ 10 │   7 │        3 │      0 │ Resolution date is  │       0s │  │
│  │                    │                      │    │     │          │        │ before seed         │          │  │
│  │                    │                      │    │     │          │        │ creation date (2),  │          │  │
│  │                    │                      │    │     │          │        │ Undetermined label  │          │  │
│  │                    │                      │    │     │          │        │ (1)                 │          │  │
│  │ QuestionRendererT… │ Complete             │  7 │   7 │        0 │      0 │ -                   │       0s │  │
│  └────────────────────┴──────────────────────┴────┴─────┴──────────┴────────┴─────────────────────┴──────────┘  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## View Results

Inspect the generated questions and answers. Each sample contains `seed`, `question`, `label`, `prompt`, and optional `context` and `meta` fields. See [API.md](../API.md) for the complete sample structure.

In [6]:
%pip install pandas

from IPython.display import clear_output
clear_output()

In [7]:
import pandas as pd

# Download samples to memory
samples = dataset.download()
rows = dataset.flattened()
df = pd.DataFrame(rows)

print(f"Generated {dataset.num_rows} samples (%.1f%% valid)\n" % (dataset.valid_count() / dataset.num_rows * 100))

cols = ["question_text", "answer", "label_confidence", "is_valid", "invalid_reason"]
df[[c for c in cols if c in df.columns]]

Generated 10 samples (70.0% valid)



,question_text,label_confidence,is_valid,invalid_reason
0,Will OpenAI expand ChatGPT Instant Checkout to...,0.90,True,NaN
1,Will Shoulder Innovations complete the full co...,1.00,True,NaN
2,Will DeepSeek release a 'next-generation' AI m...,1.00,True,NaN
3,Will Thermo Fisher Scientific announce a new a...,1.00,False,Resolution date is before seed creation date
4,Will BYD's 14.5MWh 'Haohan' battery energy sto...,0.95,False,Resolution date is before seed creation date
5,Will Lumentum begin sampling its ELSFP modules...,0.95,True,NaN
6,Will Suno Studio be available to free-tier use...,0.95,True,NaN
7,Will JCB make the new Fastrac 6000 Series trac...,1.00,False,Undetermined label
8,Will Amazon Web Services (AWS) officially laun...,1.00,True,NaN
9,Will AWS officially launch or announce the age...,0.95,True,NaN


## Using Other Answer Types

To switch answer types, swap the answer type class and adjust the question generator accordingly. Key differences:

- **Continuous**: Consider using `ForwardLookingQuestionGenerator` for numeric predictions; tailor examples to numeric outcomes
- **Multiple Choice**: Examples should include answer options (A/B/C/D format); can use `questions_per_seed` parameter
- **Free Response**: Examples should be open-ended questions requiring multi-sentence answers

In [ ]:
from lightningrod import ForwardLookingQuestionGenerator

answer_type = ContinuousAnswerType()

seed_generator = NewsSeedGenerator(
    start_date=datetime(2024, 1, 1),
    end_date=datetime(2024, 2, 1),
    search_query=[
        "NBA player statistics",
        "NBA player performance",
        "NBA game statistics",
    ],
)

question_generator = ForwardLookingQuestionGenerator(
    instructions=(
        "Write forecasting questions that require a single, specific numeric answer. "
        "Generate questions about games or events mentioned in the news articles that have already occurred and have known outcomes. "
        "Be very specific: include team names, player names, dates, or opponents when mentioned in the article. "
        "Focus on quantifiable outcomes like points scored, rebounds, assists, or other measurable statistics. "
        "Questions should reference specific games or events from the articles that have already happened so answers can be found."
    ),
    examples=[
        "How many points will LeBron James score in the Lakers' game against the Warriors on January 15th?",
        "How many rebounds will the Lakers grab in their game on January 20th?",
        "How many points will the Bulls score in their game against the Celtics?",
        "How many three-pointers will Steph Curry make in the Warriors' game on January 18th?",
    ],
    bad_examples=[
        "Will LeBron James score more than 30 points in his next game?",         # Binary, not a continuous answer
        "Which team will win the next Lakers game?",                             # Categorical, not numeric
        "How likely is it that the Bulls will score over 100 points?",           # Probability, not a specific number
        "How many points will LeBron James score in his next game?",            # Future event, can't be resolved
        "How has LeBron's scoring average changed this season?",                 # Vague, not a concrete forecast
    ],
    answer_type=answer_type,
)

pipeline_config = QuestionPipeline(
    seed_generator=seed_generator,
    question_generator=question_generator,
    labeler=WebSearchLabeler(answer_type=answer_type, confidence_threshold=0.5),
    renderer=QuestionRenderer(answer_type=answer_type),
)

# Uncomment to run:
# dataset = lr.transforms.run(pipeline_config, max_seeds=10, name="Continuous answer type")  # increase to ~5000 for a real run

In [ ]:
answer_type = MultipleChoiceAnswerType()

seed_generator = NewsSeedGenerator(
    start_date=datetime(2025, 10, 1),
    end_date=datetime(2025, 11, 1),
    search_query="technology industry news",
)

question_generator = QuestionGenerator(
    instructions=(
        "Generate multiple choice questions about technology industry events. "
        "Each question should have 3-4 answer options, with one clearly correct answer."
    ),
    examples=[
        "Question: Which company will release a new AI model first in 2025?\n"
        "A) OpenAI\n"
        "B) Google\n"
        "C) Anthropic\n"
        "Label: A",
        "Question: What will be the main focus of Apple's next product launch?\n"
        "A) AI features\n"
        "B) Battery life\n"
        "C) Camera improvements\n"
        "D) Design changes\n"
        "Label: A",
    ],
    bad_examples=[
        "Question: Will AI be important? Label: Yes",
        "Question: What is AI? Label: Artificial Intelligence",
    ],
    questions_per_seed=2,
    answer_type=answer_type,
)

pipeline_config = QuestionPipeline(
    seed_generator=seed_generator,
    question_generator=question_generator,
    labeler=WebSearchLabeler(answer_type=answer_type, confidence_threshold=0.5),
    renderer=QuestionRenderer(answer_type=answer_type),
)

# Uncomment to run:
# dataset = lr.transforms.run(pipeline_config, max_seeds=10, name="Multiple choice answer type")  # increase to ~5000 for a real run

In [ ]:
answer_type = FreeResponseAnswerType()

seed_generator = NewsSeedGenerator(
    start_date=datetime(2025, 10, 1),
    end_date=datetime(2025, 11, 1),
    search_query="technology industry trends",
)

question_generator = QuestionGenerator(
    instructions=(
        "Generate open-ended forecasting questions that require detailed explanations. "
        "Questions should ask for analysis, predictions, or explanations that require multiple sentences to answer."
    ),
    examples=[
        "What will be the main theme of Apple's next product launch?",
        "Explain the potential impact of the new AI regulation on tech companies.",
        "What trends will dominate the technology industry in 2025?",
        "Describe how the recent merger will affect the competitive landscape.",
    ],
    bad_examples=[
        "Will Apple release a new product?",
        "What is AI?",
        "When will the product launch?",
    ],
    answer_type=answer_type,
)

pipeline_config = QuestionPipeline(
    seed_generator=seed_generator,
    question_generator=question_generator,
    labeler=WebSearchLabeler(answer_type=answer_type, confidence_threshold=0.5),
    renderer=QuestionRenderer(answer_type=answer_type),
)

# Uncomment to run:
# dataset = lr.transforms.run(pipeline_config, max_seeds=10, name="Free response answer type")  # increase to ~5000 for a real run